In [2]:
dp = [0, 1, 2, 1]

dp.append(min(dp[3] + 1, dp[1] + 1))

print(dp[4])

2


In [3]:
def solution(coupons, target):
    dp = [target + 1] * (target + 1)
    dp[0] = 0

    for amount in range(1, target + 1):
        for coupon in coupons:
            if amount >= coupon:
                count = dp[amount - coupon] + 1
                dp[amount] = min(dp[amount], count)

    return dp[target]

print(solution([1, 3, 4], 6))

2


In [5]:
import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

torch.manual_seed(42)

# ① 학습 데이터 준비
# RGB 이미지 4장 (64 x 64)
X = torch.randn(4, 3, 64, 64)

# 클래스 0 또는 1
y = torch.tensor([0, 0, 1, 1])


# ② 모델 정의
def create_model(method):
    # 사전학습 가중치 설정
    if method == "scratch":
        model = resnet18(weights=None)
    else:
        model = resnet18(
            weights=ResNet18_Weights.DEFAULT
        )

    # 새로운 분류기 (2개 클래스)
    model.fc = nn.Linear(
        model.fc.in_features, 2
    )

    # Feature Extraction
    if method == "feature":

        # 기존 레이어 동결
        for param in model.parameters():
            param.requires_grad = False

        # 분류기만 학습
        for param in model.fc.parameters():
            param.requires_grad = True

    # Partial Fine-Tuning
    elif method == "partial":

        # 기존 레이어 동결
        for param in model.parameters():
            param.requires_grad = False

        # 마지막 블록 학습
        for param in model.layer4.parameters():
            param.requires_grad = True

        # 분류기 학습
        for param in model.fc.parameters():
            param.requires_grad = True

    # Scratch / Full Fine-Tuning
    # 모든 파라미터 학습 (기본 설정)
    return model


# ③ 손실 함수
criterion = nn.CrossEntropyLoss()


# ④ 네 가지 방식 학습
methods = [
    "scratch",
    "feature",
    "partial",
    "full"
]

for method in methods:

    print(f"\n===== {method} =====")

    model = create_model(method)

    # 학습 모드 설정
    if method in ("feature", "partial"):

        # 동결된 BatchNorm 통계도 고정
        model.eval()

        if method == "partial":
            model.layer4.train()

        model.fc.train()

    else:
        model.train()

    # Optimizer 설정
    optimizer = torch.optim.Adam(
        filter(
            lambda p: p.requires_grad,
            model.parameters()
        ),
        lr=0.0001
    )

    # ⑤ 기울기 초기화
    optimizer.zero_grad()

    # ⑥ 순전파
    outputs = model(X)

    # ⑦ 손실 계산
    loss = criterion(outputs, y)

    # ⑧ 역전파
    loss.backward()

    # ⑨ 가중치 업데이트
    optimizer.step()

    # 학습 파라미터 수 확인
    trainable_params = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    print("Loss:", round(loss.item(), 4))
    print("학습 파라미터:", trainable_params)

    # ⑩ 학습 후 예측
    model.eval()

    with torch.no_grad():
        predictions = model(X).argmax(dim=1)

    print("예측:", predictions.tolist())
    print("정답:", y.tolist())


===== scratch =====
Loss: 0.6063
학습 파라미터: 11177538
예측: [1, 1, 1, 1]
정답: [0, 0, 1, 1]

===== feature =====
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\mong0/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100.0%


Loss: 0.8552
학습 파라미터: 1026
예측: [0, 0, 0, 0]
정답: [0, 0, 1, 1]

===== partial =====
Loss: 0.6402
학습 파라미터: 8394754
예측: [1, 1, 1, 1]
정답: [0, 0, 1, 1]

===== full =====
Loss: 0.7962
학습 파라미터: 11177538
예측: [0, 0, 0, 0]
정답: [0, 0, 1, 1]


In [ ]:
트랜스포머
어텐션을 중심으로 여러 위치 사이의 관게를 계산하는 신경망 구조
각 위치에 따른 위치 정보를 얼마나 참고할지 계산하여 새로운 계산 구조를 만든다
구조상 인코더, 디코더를 사용하지만 한쪽만 사용하는 경우도 존재할 수 있다.
주로 텍스트, 시계열, 이미지 등 광범위하게 사용된다.
문장에서 멀리 떨어진 단어 사이의 관계 파악, 시게열에서 특정 시점 등을 예측하는 데 주로 사용된다.
학습: Q K V 정보를 어떤 기준으로 비교하고 어떻게 전달할지
학스비 FFN 위치별 특징을 비선형으로 변환
학습: 임베딩, 정규화 -> 입력 표현과 값 스케일링
